In [34]:
import pandas as pd 

df = pd.read_csv('data/main_data.csv')
df.head(3)

,city,rooms,m2,kitchen_m2,repair,district,year,floor,all_floor,real_price
0,Moscow,3,79,15,not_needs,very good,1969,5,6,39810636
1,Moscow,1,45,11,not_needs,medium,2020,2,6,18934846
2,Moscow,1,60,7,needs,good,1974,29,35,22566457


In [35]:
df['building_age'] = 2026 - df['year']
features = ['rooms', 'm2', 'kitchen_m2', 'building_age', 'floor', 'all_floor']



In [36]:
x = df.drop(columns=['real_price', 'year'])
y = df.drop(columns=['city', 'rooms', 'm2', 'kitchen_m2', 'repair', 'district', 'building_age', 'floor', 'all_floor', 'year'])
print(x, y)

        city  rooms   m2  kitchen_m2            repair   district  floor  \
0     Moscow      3   79          15         not_needs  very good      5   
1     Moscow      1   45          11         not_needs     medium      2   
2     Moscow      1   60           7             needs       good     29   
3     Moscow      4  104          14  absolutely_needs   very bad     10   
4     Moscow      3   87          12  absolutely_needs        bad     14   
...      ...    ...  ...         ...               ...        ...    ...   
7995  Samara      4  102          12         not_needs       good     32   
7996  Samara      1   43          12             needs        bad      1   
7997  Samara      2   69           9             needs  very good     27   
7998  Samara      3   89          12             needs        bad     16   
7999  Samara      2   72           8             needs        bad      7   

      all_floor  building_age  
0             6            57  
1             6        

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=25)
print(x_train)

                 city  rooms   m2  kitchen_m2            repair   district  \
549            Moscow      3   98          14             needs     medium   
5640  Nizhny Novgorod      3   95          12         not_needs        bad   
2950            Kazan      3   88          12             needs        bad   
1435       Petersburg      1   62          11             needs   very bad   
6179      Krasnoyarsk      3  101          16         not_needs  very good   
...               ...    ...  ...         ...               ...        ...   
7485           Samara      2   82          12  absolutely_needs        bad   
2191            Kazan      2   93           7             needs   very bad   
6618      Krasnoyarsk      3   99          13             needs     medium   
318            Moscow      2   81          13             needs       good   
5252  Nizhny Novgorod      5  128          16             needs  very good   

      floor  all_floor  building_age  
549      17         17  

In [38]:
from sklearn.preprocessing import StandardScaler
import numpy as np

features = ['rooms', 'm2', 'kitchen_m2', 'building_age', 'floor', 'all_floor']


y_train_log = np.log1p(y_train.values.ravel())
y_test_log  = np.log1p(y_test.values.ravel())

scaler = StandardScaler()
x_train[features] = scaler.fit_transform(x_train[features])     
x_test[features] = scaler.transform(x_test[features])          

In [39]:
from catboost import Pool

cat_features = ['city', 'repair', 'district']

train_pool = Pool(
    data = x_train,
    label = y_train_log,
    cat_features = cat_features, 
    feature_names = x_train.columns.tolist()
)

test_pool = Pool(
    data = x_test,
    label = y_test_log,
    cat_features = cat_features,       
    feature_names = x_test.columns.tolist()
)

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import joblib


model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=10,
    loss_function='RMSE',           
    eval_metric='RMSE',
    random_seed=25,
    verbose=200,
    early_stopping_rounds = 200
)


model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True
)


y_pred_log = model.predict(x_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"MAE:   {mae:,.0f} ₽")
print(f"RMSE:  {rmse:,.0f} ₽")
print(f"MAPE:  {mape:.2f}%")
print(f"R²:    {r2:.4f}")

model.save_model("models/kvartis_model.cbm")
joblib.dump(scaler, "models/kvartis_scaler.pkl")
print("модель сохранина в папку models")

0:	learn: 0.5513545	test: 0.5469146	best: 0.5469146 (0)	total: 40.9ms	remaining: 2m 2s
200:	learn: 0.0938246	test: 0.0924265	best: 0.0924265 (200)	total: 9.19s	remaining: 2m 8s
400:	learn: 0.0799514	test: 0.0861052	best: 0.0861052 (400)	total: 18.3s	remaining: 1m 58s
600:	learn: 0.0705341	test: 0.0848817	best: 0.0848743 (596)	total: 28s	remaining: 1m 51s
800:	learn: 0.0632115	test: 0.0847065	best: 0.0846976 (714)	total: 37.4s	remaining: 1m 42s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.08469757416
bestIteration = 714

Shrink model to first 715 iterations.
MAE:   993,473 ₽
RMSE:  1,729,199 ₽
MAPE:  7.14%
R²:    0.9605
модель сохранина в папку models
